In [1]:
import sqlite3
import pandas as pd

conn_acc_ver = sqlite3.connect('data/BikeToDrive_1_Accessoireverkoop.db')
conn_fiets_ver = sqlite3.connect('data/BikeToDrive_2_Fietsverkoop.db')
conn_onderhoud = sqlite3.connect('data/BikeToDrive_3_Onderhoud.db')
conn_acc_inkoop = sqlite3.connect('data/BikeToDrive_4_Accessoire_inkoop.db')
conn_fiets_inkoop = sqlite3.connect('data/BikeToDrive_5_Fiets_inkoop.db')
db_naam = 'data/SDM_DEAI.db'
conn_sdm = sqlite3.connect(db_naam)

In [2]:
def reset_sdm(conn):
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = OFF;")
    
    # We halen nu dynamisch alle tabellen op om ze te legen
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tabellen = cursor.fetchall()
    
    print("Start met het legen van het SDM...")
    for tabel_tuple in tabellen:
        tabel = tabel_tuple[0]
        if tabel != 'sqlite_sequence':
            try:
                cursor.execute(f"DELETE FROM {tabel};")
                print(f" - Tabel '{tabel}' is leeggemaakt.")
            except Exception as e:
                print(f"Waarschuwing bij legen van tabel '{tabel}': {e}")
                
    cursor.execute("PRAGMA foreign_keys = ON;")
    conn.commit()
    print("Reset voltooid: Alle SDM-tabellen zijn succesvol leeggemaakt.\n")

# Voer reset uit
reset_sdm(conn_sdm)

Start met het legen van het SDM...
 - Tabel 'Accessoire_Inkoop_Leverancier' is leeggemaakt.
 - Tabel 'Accessoire_Inkoop_Accessoire' is leeggemaakt.
 - Tabel 'Accessoire_Inkoop' is leeggemaakt.
 - Tabel 'Fiets_Inkoop_Fabrikant' is leeggemaakt.
 - Tabel 'Fiets_Inkoop_Fiets' is leeggemaakt.
 - Tabel 'Fiets_Inkoop' is leeggemaakt.
 - Tabel 'Onderhoud_Fabrikant' is leeggemaakt.
 - Tabel 'Onderhoud_Fiets' is leeggemaakt.
 - Tabel 'Onderhoud_Filiaal' is leeggemaakt.
 - Tabel 'Onderhoud_Monteur' is leeggemaakt.
 - Tabel 'Onderhoud' is leeggemaakt.
 - Tabel 'Fietsverkoop_Filiaal' is leeggemaakt.
 - Tabel 'Fietsverkoop_Fabrikant' is leeggemaakt.
 - Tabel 'Fietsverkoop_Monteur' is leeggemaakt.
 - Tabel 'Fietsverkoop_Klant' is leeggemaakt.
 - Tabel 'Fietsverkoop_Fiets' is leeggemaakt.
 - Tabel 'Fietsverkoop_Fiets_Verkoop' is leeggemaakt.
 - Tabel 'Accessoireverkoop_Filiaal' is leeggemaakt.
 - Tabel 'Accessoireverkoop_Leverancier' is leeggemaakt.
 - Tabel 'Accessoireverkoop_Monteur' is leeggemaakt.

In [3]:
def laad_data_in_sdm():
    print("Start met het overzetten van data (1-op-1 copy)...")
    
    # --- 1. Accessoire-inkoop ---
    print(" -> Data laden uit Accessoire-inkoop...")
    pd.read_sql("SELECT * FROM Leverancier", conn_acc_inkoop).to_sql('Accessoire_Inkoop_Leverancier', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Accessoire", conn_acc_inkoop).to_sql('Accessoire_Inkoop_Accessoire', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Accessoire_Inkoop", conn_acc_inkoop).to_sql('Accessoire_Inkoop', conn_sdm, if_exists='append', index=False)

    # --- 2. Fietsinkoop ---
    print(" -> Data laden uit Fietsinkoop...")
    pd.read_sql("SELECT * FROM Fabrikant", conn_fiets_inkoop).to_sql('Fiets_Inkoop_Fabrikant', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Fiets", conn_fiets_inkoop).to_sql('Fiets_Inkoop_Fiets', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Fiets_Inkoop", conn_fiets_inkoop).to_sql('Fiets_Inkoop', conn_sdm, if_exists='append', index=False)

    # --- 3. Onderhoud ---
    print(" -> Data laden uit Onderhoud...")
    pd.read_sql("SELECT * FROM Fabrikant", conn_onderhoud).to_sql('Onderhoud_Fabrikant', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Fiets", conn_onderhoud).to_sql('Onderhoud_Fiets', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Filiaal", conn_onderhoud).to_sql('Onderhoud_Filiaal', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Monteur", conn_onderhoud).to_sql('Onderhoud_Monteur', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Onderhoud", conn_onderhoud).to_sql('Onderhoud', conn_sdm, if_exists='append', index=False)

    # --- 4. Fietsverkoop ---
    print(" -> Data laden uit Fietsverkoop...")
    pd.read_sql("SELECT * FROM Filiaal", conn_fiets_ver).to_sql('Fietsverkoop_Filiaal', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Fabrikant", conn_fiets_ver).to_sql('Fietsverkoop_Fabrikant', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Monteur", conn_fiets_ver).to_sql('Fietsverkoop_Monteur', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Klant", conn_fiets_ver).to_sql('Fietsverkoop_Klant', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Fiets", conn_fiets_ver).to_sql('Fietsverkoop_Fiets', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Fiets_Verkoop", conn_fiets_ver).to_sql('Fietsverkoop_Fiets_Verkoop', conn_sdm, if_exists='append', index=False)

    # --- 5. Accessoireverkoop ---
    print(" -> Data laden uit Accessoireverkoop...")
    pd.read_sql("SELECT * FROM Filiaal", conn_acc_ver).to_sql('Accessoireverkoop_Filiaal', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Leverancier", conn_acc_ver).to_sql('Accessoireverkoop_Leverancier', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Monteur", conn_acc_ver).to_sql('Accessoireverkoop_Monteur', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Klant", conn_acc_ver).to_sql('Accessoireverkoop_Klant', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Accessoire", conn_acc_ver).to_sql('Accessoireverkoop_Accessoire', conn_sdm, if_exists='append', index=False)
    pd.read_sql("SELECT * FROM Accessoire_Verkoop", conn_acc_ver).to_sql('Accessoireverkoop_Accessoire_Verkoop', conn_sdm, if_exists='append', index=False)

    print("\nData succesvol geladen in het geprefixte Source Data Model!")

# Voer het inladen uit
laad_data_in_sdm()

# ==========================================
# Connecties sluiten
# ==========================================
conn_acc_ver.close()
conn_fiets_ver.close()
conn_onderhoud.close()
conn_acc_inkoop.close()
conn_fiets_inkoop.close()
conn_sdm.close()
print("Alle database connecties zijn gesloten.")

Start met het overzetten van data (1-op-1 copy)...
 -> Data laden uit Accessoire-inkoop...
 -> Data laden uit Fietsinkoop...
 -> Data laden uit Onderhoud...
 -> Data laden uit Fietsverkoop...
 -> Data laden uit Accessoireverkoop...

Data succesvol geladen in het geprefixte Source Data Model!
Alle database connecties zijn gesloten.
